In [ ]:
import pandas as pd

def calcular_custo_pitstop(csv_entrada, csv_saida):
    print("Calculando o Custo Real do Pit Stop")
    df = pd.read_csv(csv_entrada, sep=';', decimal=',')
    
    # Isola apenas as voltas de entrada nos boxes
    in_laps = df[df['Tipo_Volta'] == 'In-Lap'].copy()
    resultados = []

    for index, in_lap in in_laps.iterrows():
        carro = in_lap['Carro']
        stint_atual = in_lap['Stint']
        volta_in = in_lap['Lap']
        tempo_in = in_lap['Lap Tm (Segundos)']
        
        # A Out-Lap é a primeira volta do próximo Stint (Stint + 1)
        out_lap_df = df[(df['Carro'] == carro) & (df['Stint'] == stint_atual + 1) & (df['Tipo_Volta'] == 'Out-Lap')]
        
        if not out_lap_df.empty:
            tempo_out = out_lap_df.iloc[0]['Lap Tm (Segundos)']
            
            # Define o Ritmo Base (Mediana das voltas limpas do stint que estava acabando)
            push_laps = df[(df['Carro'] == carro) & (df['Stint'] == stint_atual) & (df['Tipo_Volta'] == 'Push') & (~df['Outlier'])]
            
            if not push_laps.empty:
                ritmo_base = push_laps['Lap Tm (Segundos)'].median()
                
                # A Fórmula do Custo Real
                custo_real = (tempo_in + tempo_out) - (2 * ritmo_base)
                
                resultados.append({
                    'Carro': carro,
                    'Volta_Parada': volta_in,
                    'Ritmo_Base': round(ritmo_base, 3),
                    'Tempo_In_Lap': round(tempo_in, 3),
                    'Tempo_Out_Lap': round(tempo_out, 3),
                    'Custo_Real_Segundos': round(custo_real, 3)
                })
    
    df_pit_cost = pd.DataFrame(resultados)
    df_pit_cost.to_csv(csv_saida, index=False, sep=';', decimal=',')
    print(f"Análise de Pit Stop concluída! Relatório salvo em: {csv_saida}")

# --- ÁREA DE EXECUÇÃO ---
arquivo_estrategia = '../data/03_processed/TELEMETRIA_ESTRATEGIA_P1.csv'
arquivo_relatorio_pit = '../data/03_processed/RELATORIO_PIT_COST_P1.csv'

calcular_custo_pitstop(arquivo_estrategia, arquivo_relatorio_pit)

⏱️ Cronômetro na mão: Calculando o Custo Real do Pit Stop...
🛑 Análise de Pit Stop concluída! Relatório salvo em: ../data/03_processed/RELATORIO_PIT_COST_P1.csv
